# Notebook 3 (Account 1): Grouped 5-fold cross-validation, **YOLOv8N only**

**On this account, change only `FOLD` (0, 1, 2, 3, 4), one run at a time. The model is already set.**
EAAI-26-20775 revision. Answers Reviewer 4.3 (stability across splits) and gives a fair model comparison for Reviewers 1.4 / 2.5.

**How it works**
- All 1,506 images are grouped by original capture (6 images per capture, 251 captures), using the filename numbering.
- Captures are split into 5 folds, balanced by class. **All 6 images of a capture always stay in the same fold**, so there is no leakage.
- The fold assignment is fixed and identical on every account. The notebook prints a check code; it must read the same everywhere.
- Each run trains one model on 4 folds and tests on the 5th, with exactly the baseline training settings.

**One run = one (MODEL, FOLD) pair, about 1.5 h on a T4.** 10 runs in total:
`yolov8n` folds 0–4 and `yolov10n` folds 0–4. You can run them on different Google accounts at the same time.

**Each account needs only** `dataset_clean.zip` at `MyDrive/fabric/dataset_clean.zip`.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Set MODEL and FOLD below, then Runtime → Run all.

In [ ]:
# ---- Settings: change MODEL and FOLD for each run ----
MODEL = "yolov8n"          # fixed for Account 1; do not change
FOLD = 0                 # <-- change this for each run: 0, 1, 2, 3, 4
DRIVE_ROOT = "/content/drive/MyDrive/fabric"
DATASET_ZIP = f"{DRIVE_ROOT}/dataset_clean.zip"
OUT_DIR = f"{DRIVE_ROOT}/revision/cv"
WORK = "/content/data"
DEVICE = 0
LOCAL_TEST = False       # leave False on Colab
EPOCHS_OVERRIDE = None   # leave None

N_FOLDS, SPLIT_SEED, IMGSZ = 5, 2026, 768
assert MODEL in ("yolov8n", "yolov10n") and FOLD in range(N_FOLDS)

# Baseline training settings (copied from the original run's args.yaml; do not change)
BASE = dict(epochs=100, patience=20, batch=16, optimizer="AdamW", lr0=0.001, lrf=0.01, cos_lr=True,
            momentum=0.937, weight_decay=0.0005, warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1,
            box=7.5, cls=0.5, dfl=1.5, seed=0, deterministic=True, amp=True, workers=8,
            mosaic=1.0, close_mosaic=10, mixup=0.1, fliplr=0.5, flipud=0.0,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0)
NAMES = {0: "Holes", 1: "Abrasion_Mark", 2: "Oil_Stain", 3: "Missing_Yarn"}

In [ ]:
import os, sys, subprocess
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    import torch
    assert torch.cuda.is_available(), "No GPU in this session. Runtime → Change runtime type → T4 GPU, then Run all again."
assert os.path.exists(DATASET_ZIP), f"Not found: {DATASET_ZIP}"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---- Unzip and build the fixed, class-balanced, capture-level folds ----
import zipfile, glob, re, random, hashlib, json, yaml
from collections import defaultdict, Counter
if not glob.glob(f"{WORK}/**/images/train", recursive=True):
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(WORK)
root = os.path.dirname(os.path.dirname(sorted(glob.glob(f"{WORK}/**/images/train", recursive=True))[0]))
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

images = []  # (abs image path, abs label path, capture id, class prefix)
for split in ("train", "val"):
    for f in sorted(os.listdir(f"{root}/images/{split}")):
        if not f.lower().endswith(IMG_EXT):
            continue
        m = re.match(r"^(.+)_(\d+)$", os.path.splitext(f)[0])
        assert m, f"Unexpected filename: {f}"
        prefix, num = m.group(1), int(m.group(2))
        images.append((f"{root}/images/{split}/{f}", f"{root}/labels/{split}/{os.path.splitext(f)[0]}.txt",
                       f"{prefix}#{(num - 1) // 6:03d}", prefix))
caps = defaultdict(list)
for im in images:
    caps[im[2]].append(im)
by_class = defaultdict(list)
for cid in sorted(caps):
    by_class[cid.split("#")[0]].append(cid)
fold_of = {}
rng = random.Random(SPLIT_SEED)
for prefix in sorted(by_class):
    ids = by_class[prefix][:]
    rng.shuffle(ids)
    for i, cid in enumerate(ids):
        fold_of[cid] = i % N_FOLDS
check = hashlib.md5(json.dumps(sorted(fold_of.items())).encode()).hexdigest()[:8]
print(f"Images: {len(images)}  captures: {len(caps)}  (expected 1506 / 251)")
print("Captures per fold:", dict(sorted(Counter(fold_of.values()).items())))
print("FOLD CHECK CODE:", check, " ← must be identical on every account")
assert len(images) == 1506 and len(caps) == 251, "Dataset differs from the manuscript; stop and report."

fold_dir = f"/content/cv_fold{FOLD}"
for split in ("train", "val"):
    for kind in ("images", "labels"):
        os.makedirs(f"{fold_dir}/{kind}/{split}", exist_ok=True)
n_split = Counter()
for img, lbl, cid, _ in images:
    split = "val" if fold_of[cid] == FOLD else "train"
    n_split[split] += 1
    for src, kind in ((img, "images"), (lbl, "labels")):
        dst = f"{fold_dir}/{kind}/{split}/{os.path.basename(src)}"
        if os.path.exists(src) and not os.path.exists(dst):
            os.symlink(src, dst)
print(f"Fold {FOLD}: train {n_split['train']} images, val {n_split['val']} images")
data_yaml = f"{fold_dir}/data.yaml"
yaml.safe_dump({"path": fold_dir, "train": "images/train", "val": "images/val", "names": NAMES}, open(data_yaml, "w"), sort_keys=False)

In [ ]:
# ---- Train (resumes automatically after a disconnect) ----
import ultralytics
from ultralytics import YOLO
run_name = f"{MODEL}_fold{FOLD}"
run_dir = f"{OUT_DIR}/{run_name}"
last, best, done_flag = f"{run_dir}/weights/last.pt", f"{run_dir}/weights/best.pt", f"{run_dir}/TRAINING_DONE"
args = dict(BASE, data=data_yaml, imgsz=IMGSZ, device=DEVICE, project=OUT_DIR, name=run_name, exist_ok=True, pretrained=True)
if EPOCHS_OVERRIDE:
    args["epochs"] = EPOCHS_OVERRIDE
print("Ultralytics", ultralytics.__version__, "| run", run_name)
if os.path.exists(done_flag):
    print("Training already finished.")
elif os.path.exists(last):
    print("Resuming from", last)
    YOLO(last).train(resume=True)
else:
    YOLO(f"{MODEL}.pt").train(**args)
open(done_flag, "w").write("done")

In [ ]:
# ---- Evaluate on the held-out fold at 640 (deployed size) and 768 ----
model = YOLO(best)
def metrics_dict(r):
    b = r.box
    out = {"mAP50": float(b.map50), "mAP50_95": float(b.map), "P": float(b.mp), "R": float(b.mr), "per_class": {}}
    for i, c in enumerate(b.ap_class_index):
        p, rc, ap50, ap = b.class_result(i)
        out["per_class"][NAMES[int(c)]] = {"P": round(float(p), 4), "R": round(float(rc), 4), "AP50": round(float(ap50), 4), "AP50_95": round(float(ap), 4)}
    return out
evals = {}
for sz in (640, 768):
    r = model.val(data=data_yaml, imgsz=sz, batch=16, device=DEVICE, split="val", plots=False,
                  project=run_dir, name=f"val_{sz}", exist_ok=True, verbose=False)
    evals[f"infer{sz}"] = metrics_dict(r)
    m = evals[f"infer{sz}"]
    print(f"{run_name} @ {sz}: mAP50 {m['mAP50']:.3f}  mAP50-95 {m['mAP50_95']:.3f}  P {m['P']:.3f}  R {m['R']:.3f}")

# ---- Save all predictions on the held-out fold (for threshold analysis chosen out-of-sample) ----
import csv
val_dir = f"{fold_dir}/images/val"
files = sorted(os.listdir(val_dir))
with open(f"{run_dir}/val_predictions.csv", "w", newline="") as fp, open(f"{run_dir}/val_ground_truth.csv", "w", newline="") as fg:
    wp, wg = csv.writer(fp), csv.writer(fg)
    wp.writerow(["image", "cls", "conf", "xc", "yc", "w", "h"]); wg.writerow(["image", "cls", "xc", "yc", "w", "h"])
    for i in range(0, len(files), 16):
        chunk = files[i:i + 16]
        for f, res in zip(chunk, model.predict([f"{val_dir}/{f}" for f in chunk], imgsz=640, conf=0.001, iou=0.7, device=DEVICE, verbose=False)):
            for c, s, b in zip(res.boxes.cls.tolist(), res.boxes.conf.tolist(), res.boxes.xywhn.tolist()):
                wp.writerow([f, int(c), round(s, 5)] + [round(x, 6) for x in b])
        for f in chunk:
            lf = f"{fold_dir}/labels/val/{os.path.splitext(f)[0]}.txt"
            if os.path.exists(lf):
                for line in open(lf):
                    q = line.split()
                    if len(q) == 5:
                        wg.writerow([f, int(q[0])] + q[1:])

summary = {"run": run_name, "model": MODEL, "fold": FOLD, "fold_check": check, "n_train": n_split["train"], "n_val": n_split["val"],
           "ultralytics": ultralytics.__version__, "eval": evals}
json.dump(summary, open(f"{run_dir}/summary.json", "w"), indent=2)
print("\n=== PASTE THE LINE BELOW INTO THE CHAT ===")
print(json.dumps(summary, separators=(",", ":")))
print(f"\nAlso upload from Drive ({OUT_DIR}/{run_name}/): val_predictions.csv and val_ground_truth.csv")